In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import glob
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc

In [2]:
in_dir = "../../results/02_tfbs_prediction/05_eval"
out_dir = "../../results/02_tfbs_prediction/06_plot_curve"
os.makedirs(out_dir, exist_ok=True)

In [3]:
recall_threshold = 0.2

In [4]:
# fixed color per method so all TFs use the same scheme
method_colors = {
    "accessatac_both":  "#1f77b4",
    "accessatac_dddss": "#ff7f0e",
    "accessatac_tn5":   "#2ca02c",
    "seq":              "#d62728",
}
method_order = list(method_colors.keys())

In [5]:
for cell_type in ["K562", "HepG2"]:
    os.makedirs(f"{out_dir}/{cell_type}", exist_ok=True)
 
    all_files = glob.glob(
        f"../../results/02_tfbs_prediction/04_train/{cell_type}/seq/prediction/*.csv")
    tfs = [os.path.basename(f).replace(".csv", "") for f in all_files]
 
    df_res = pd.read_csv(f"{in_dir}/{cell_type}/precision_at_recall.csv")
    df_res = df_res[df_res["target_recall"] == recall_threshold]
 
    for tf in tfs:
        out_png = f"{out_dir}/{cell_type}/{tf}_precision_recall_curve.png"
        # if os.path.exists(out_png):
        #     continue
 
        df = pd.read_csv(f"{in_dir}/{cell_type}/{tf}_precision_recall.csv")
        df = df[df["recall"] <= recall_threshold]
        df = df.sort_values(by=["method", "recall"])
        df = df.drop_duplicates(subset=["recall", "method"], keep="last")
 
        _df_res = df_res[df_res["tf"] == tf]
 
        fig, ax = plt.subplots(figsize=(5.5, 4))
 
        # plot each method with interpolated (monotonic) precision for smoothness
        for method in method_order:
            sub = df[df["method"] == method].sort_values("recall")
            if sub.empty:
                continue
            rec = sub["recall"].values
            prec = sub["precision"].values

            # auPRC-at-recall value for the legend
            val = _df_res.loc[_df_res["method"] == method, "precision"]
            label = f"{method} ({val.values[0]:.2f})" if len(val) else method
 
            ax.plot(rec, prec, color=method_colors[method],
                    lw=1.4, label=label)
 
        ax.set_xlim(0, recall_threshold)
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.set_title(f"{tf} ({cell_type})", fontsize=11)
        ax.legend(frameon=False, fontsize=9, loc="lower left",
                  title="precision @ recall")
        fig.tight_layout()
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        plt.close(fig)